# TF-IDF Baseline & Initial Risk Rules (Naeem)

Extracted from the team's shared Colab notebook (`Real_EDA.ipynb`, section *Naeem - Baseline Model & Initial Risk Rules*).

- **Task #4:** TF-IDF + Logistic Regression baseline with per-category precision / recall / F1 on the 7 categories selected during team EDA
- **Task #5:** Initial risk tier rules (High / Medium / Low with weights and justifications) for those categories
- Combined view of baseline performance vs. risk tier, plus error analysis notes

In [ ]:
!pip install -q huggingface_hub scikit-learn

In [ ]:
# Task #4: Establish a TF-IDF Baseline with Per-Category Metrics
import json
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

# 1. Download and load CUAD_v1.json
print("Loading CUAD_v1.json...")
json_file = hf_hub_download(
    repo_id="theatticusproject/cuad",
    filename="CUAD_v1/CUAD_v1.json",
    repo_type="dataset"
)

with open(json_file, "r") as f:
    cuad_data = json.load(f)["data"]

# 2. Extract contract texts and build binary labels per category
contracts = []
clause_rows = []

for doc in cuad_data:
    title = doc["title"]
    context = doc["paragraphs"][0]["context"]
    contracts.append({"title": title, "text": context})

    for qa in doc["paragraphs"][0]["qas"]:
        # Extract normalized category from question text
        # Format in CUAD: 'Highlight the parts (if any) of this contract related to "Category Name"...'
        q_text = qa.get("question", "")
        if '"' in q_text:
            category = q_text.split('"')[1]
        else:
            category = qa["id"].split("_", 1)[1].split("$")[0].replace("_", " ")

        has_clause = not qa.get("is_impossible", False) and len(qa.get("answers", [])) > 0
        clause_rows.append({
            "title": title,
            "category": category.strip(),
            "has_clause": int(has_clause)
        })

docs_df = pd.DataFrame(contracts)
clauses_df = pd.DataFrame(clause_rows)

# 3. Official Atticus Project split: First 408 train, remaining 102 test
train_docs = docs_df.iloc[:408].copy().reset_index(drop=True)
test_docs = docs_df.iloc[408:].copy().reset_index(drop=True)

print(f"Loaded {len(train_docs)} train contracts and {len(test_docs)} test contracts.")

# 4. TF-IDF vectorization on train contracts only
tfidf = TfidfVectorizer(max_features=10000, stop_words="english", ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_docs["text"])
X_test = tfidf.transform(test_docs["text"])

# 5. Target categories analyzed during team EDA
target_categories = [
    "Anti-Assignment",
    "Change Of Control",
    "Insurance",
    "Renewal Term",
    "Notice Period To Terminate Renewal",
    "Audit Rights",
    "Termination For Convenience"
]

results = []

for cat in target_categories:
    cat_df = clauses_df[clauses_df["category"] == cat]

    # Map binary target directly
    train_map = cat_df.groupby("title")["has_clause"].max()
    test_map = cat_df.groupby("title")["has_clause"].max()

    y_train = train_docs["title"].map(train_map).fillna(0).astype(int).values
    y_test = test_docs["title"].map(test_map).fillna(0).astype(int).values

    # Train Logistic Regression baseline
    if len(np.unique(y_train)) < 2:
        prec, rec, f1 = 0.0, 0.0, 0.0
    else:
        model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

    results.append({
        "Category": cat,
        "Train Pos": int(y_train.sum()),
        "Test Pos": int(y_test.sum()),
        "Precision": round(float(prec), 3),
        "Recall": round(float(rec), 3),
        "F1_Score": round(float(f1), 3)
    })

baseline_results_df = pd.DataFrame(results)
display(baseline_results_df)

Loading CUAD_v1.json...
Loaded 408 train contracts and 102 test contracts.


,Category,Train Pos,Test Pos,Precision,Recall,F1_Score
0,Anti-Assignment,304,70,0.783,0.929,0.850
1,Change Of Control,97,24,0.381,0.333,0.356
2,Insurance,133,33,0.545,0.727,0.623
3,Renewal Term,139,37,0.606,0.541,0.571
4,Notice Period To Terminate Renewal,90,21,0.368,0.333,0.350
5,Audit Rights,172,42,0.667,0.571,0.615
6,Termination For Convenience,150,33,0.447,0.515,0.479


In [ ]:
# =====================================================================
# Task #5: Initial Risk Assessment Rules & Tier Mapping
# =====================================================================

risk_framework = [
    {
        "Category": "Change Of Control",
        "Risk_Tier": "High",
        "Weight": 3,
        "Justification": "Can trigger immediate termination, accelerated obligations, or forfeiture of IP rights upon acquisition or merger."
    },
    {
        "Category": "Termination For Convenience",
        "Risk_Tier": "High",
        "Weight": 3,
        "Justification": "Allows counterparty to exit without breach, leading to unexpected revenue loss and unrecoverable upfront capital."
    },
    {
        "Category": "Anti-Assignment",
        "Risk_Tier": "Medium",
        "Weight": 2,
        "Justification": "Restricts corporate restructuring, sale of assets, or transferring obligations to subsidiaries without prior written consent."
    },
    {
        "Category": "Notice Period To Terminate Renewal",
        "Risk_Tier": "Medium",
        "Weight": 2,
        "Justification": "Missing short notification windows automatically locks the organization into unwanted renewal terms and added costs."
    },
    {
        "Category": "Renewal Term",
        "Risk_Tier": "Medium",
        "Weight": 2,
        "Justification": "Defines duration and extension terms; critical for operational planning but standard in commercial agreements."
    },
    {
        "Category": "Insurance",
        "Risk_Tier": "Low",
        "Weight": 1,
        "Justification": "Standard compliance requirement; easily mitigated through standard commercial policy verification."
    },
    {
        "Category": "Audit Rights",
        "Risk_Tier": "Low",
        "Weight": 1,
        "Justification": "Standard oversight mechanism; poses minimal risk assuming regular financial and operational record-keeping."
    }
]

risk_rules_df = pd.DataFrame(risk_framework)
display(risk_rules_df)

,Category,Risk_Tier,Weight,Justification
0,Change Of Control,High,3,"Can trigger immediate termination, accelerated..."
1,Termination For Convenience,High,3,"Allows counterparty to exit without breach, le..."
2,Anti-Assignment,Medium,2,"Restricts corporate restructuring, sale of ass..."
3,Notice Period To Terminate Renewal,Medium,2,Missing short notification windows automatical...
4,Renewal Term,Medium,2,Defines duration and extension terms; critical...
5,Insurance,Low,1,Standard compliance requirement; easily mitiga...
6,Audit Rights,Low,1,Standard oversight mechanism; poses minimal ri...


In [ ]:
# Combine Baseline Evaluation with Risk Framework
combined_analysis_df = baseline_results_df.merge(
    risk_rules_df[["Category", "Risk_Tier", "Weight"]],
    on="Category",
    how="left"
)

display(combined_analysis_df.sort_values(by=["Weight", "F1_Score"], ascending=[False, True]))

,Category,Train Pos,Test Pos,Precision,Recall,F1_Score,Risk_Tier,Weight
1,Change Of Control,97,24,0.381,0.333,0.356,High,3
6,Termination For Convenience,150,33,0.447,0.515,0.479,High,3
4,Notice Period To Terminate Renewal,90,21,0.368,0.333,0.350,Medium,2
3,Renewal Term,139,37,0.606,0.541,0.571,Medium,2
0,Anti-Assignment,304,70,0.783,0.929,0.850,Medium,2
5,Audit Rights,172,42,0.667,0.571,0.615,Low,1
2,Insurance,133,33,0.545,0.727,0.623,Low,1


### Baseline Performance Gaps & Error Analysis (Task #4 Findings)

1. **Class Imbalance Impact:**
   * Categories with low positive prevalence in the training set suffer from lower precision and unstable recall.
   * Applying balanced class weights helped prevent minority categories from being completely predicted as 0, but false positives remain an issue.

2. **Full-Document Context Limitations:**
   * Whole-contract TF-IDF treats thousands of tokens as a single bag-of-words. Important short clauses (such as 30-day notice periods) get diluted across 50,000+ characters of legal boilerplate.
   * This confirms the need for our October milestone: chunk-based tokenization paired with fine-tuned transformer encoders (like DistilRoBERTa) to isolate local sentence context.

3. **Risk Scoring Relevance (Task #5 Integration):**
   * High-risk clauses like `Change Of Control` and `Termination For Convenience` require high recall to ensure potential operational risks are not missed during contract triage.